# Chargement et exploration des données d'open food facts

# 1. Chargement du fichier .parquet

In [33]:
# Installation et import des modules nécessaires pour le parcours du fichier food.parquet
# %pip install pandas pyarrow duckdb

import pandas as pd
import pyarrow as pa
import duckdb

# Découverte du schéma du fichier food.parquet et comptage des produits vendus en France
print(duckdb.sql("DESCRIBE SELECT countries_tags FROM '../data/food.parquet'"))
print(duckdb.sql("SELECT COUNT() FROM '../data/food.parquet' WHERE countries_tags == '[\"en:france\"]'"))
print(duckdb.sql("SELECT COUNT() FROM '../data/food.parquet' WHERE 'en:france' IN countries_tags"))

┌────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │ column_type │  null   │   key   │ default │  extra  │
│    varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ countries_tags │ VARCHAR[]   │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      1135260 │
└──────────────┘

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      1249256 │
└──────────────┘



En analysant certaines colonnes et certaines valeurs de la base de données sur ce lien qui permet de parcourir la base : https://huggingface.co/datasets/openfoodfacts/product-database, je peux apercevoir 3 colonnes intéressantes sur lesquelles se baser pour trouver et filtrer sur les produits vendus en France : 
- purchase_places_tags : il semblerait que ce soit la colonne contenant une liste des lieux d'achats du produit.
- lang : correspond à la langue dans laquelle la fiche produit a été renseignée. Cette colonne n'a pas été retenue, car elle indique uniquement la langue utilisée par le contributeur et non le pays où le produit est réellement vendu. Par exemple, un produit rédigé en français peut être commercialisé en Belgique, au Canada ou en Suisse.
- countries_tags : contient la liste des pays dans lesquels le produit est référencé ou disponible. Cette colonne a été retenue, car elle est la plus complète et a l'air de représenter les pays de commercialisation du produit.

Dans un premier temps, un filtre utilisant une égalité stricte (`countries_tags == ["en:france"]`) a été appliqué. Cependant, cette approche ne retenait que les produits référencés **uniquement** en France. Tous les produits également commercialisés dans d'autres pays, par exemple `["en:france", "en:belgium"]`, étaient donc exclus à tort. Ce filtrage aboutissait à un total de **1 133 372 produits**, ce qui sous-estimait le nombre réel de produits vendus en France.

Cependant la colonne countries_tags est de type VARCHAR[], et contient donc une liste. Il est donc plus judicieux d'utiliser l'opérateur "IN" plutôt qu'une égalité stricte.
C'est donc pourquoi nous avons remplacé (`countries_tags == ["en:france"]`) par (`'en:france' in countries_tags`). Cela permet de sélectionner tous les produits dont la France fait partie des pays de commercialisation, qu'ils soient vendus uniquement en France ou dans plusieurs pays. Le résultat obtenu est de **1 249 256 produits** référencés comme vendus en France, un chiffre cohérent avec celui obtenu par mes collègues.

In [34]:
food_df = pd.read_parquet(
    "../data/food.parquet",
    engine="pyarrow",
    filters=[("countries_tags", "IN", "en:france")],
    columns=["brands", "purchase_places_tags", "lang", "countries_tags"]
)

ValueError: "countries_tags" is not a valid operator in predicates.

Pourquoi utiliser DuckDB plutôt que pd.read_parquet(filters=...) ?

Le paramètre filters de pandas.read_parquet() permet uniquement d'effectuer des comparaisons simples sur les valeurs d'une colonne, comme ==, !=, <, >, in ou not in. Ces opérateurs sont adaptés aux colonnes contenant une valeur unique par ligne, mais ils ne permettent pas de vérifier si une valeur est présente dans une liste stockée dans une colonne.

Dans notre cas, la colonne countries_tags (comme purchase_places_tags) est de type VARCHAR[] : un même produit peut être référencé dans plusieurs pays. L'objectif n'était donc pas de comparer la colonne à une valeur unique, mais de vérifier si "en:france" est présent dans la liste des pays associés à chaque produit. Ce type de test n'est pas pris en charge par la méthode read_parquet et son paramètre filters.

DuckDB gère en revanche nativement les colonnes de type liste. Il permet d'utiliser des expressions telles que 'en:france' IN countries_tags (ou la fonction list_contains) pour tester directement l'appartenance d'une valeur à une liste.

L'utilisation de DuckDB présente ainsi deux avantages :

- elle permet de filtrer correctement les colonnes contenant des listes, ce qui n'est pas possible avec pd.read_parquet(filters=...)
- elle exécute directement la requête SQL sur le fichier .parquet, sans avoir à charger l'ensemble des données en mémoire avant le filtrage, ce qui améliore les performances.

Pour ces raisons, le filtrage des produits vendus en France a été réalisé avec duckdb.sql(...) plutôt qu'avec le paramètre filters de pd.read_parquet().

In [39]:
# Chargement du fichier food.parquet dans une DataFrame pandas en filtrant les produits vendus en France
food_df = duckdb.sql("""
    SELECT brands, purchase_places_tags, lang, countries_tags, code, created_t
    FROM '../data/food.parquet'
    WHERE 'en:france' in countries_tags
""").df()

In [ ]:
#1 combien de produits vendus en France ? -Sacha
print(f"Il y a {food_df.shape[0]} produits alimentaires vendus en France dans le fichier food.parquet.")

Il y a 1249256 produits alimentaires vendus en France dans le fichier food.parquet.


In [ ]:
#2 quelle  part a un Nutri-Score renseigné? -Clement

In [ ]:
#3 les dix marques les plus présentes ? -Sacha
top10_marques = (
    food_df[food_df["brands"].notna() & (food_df["brands"] != "")]
    ["brands"]
    .value_counts()
    .head(10)
)

top10_marques

brands
CraftBoss             1785721957
Loiret & haëntjens    1785708704
Joyart                1785698103
Saboremsa             1785692294
Globus                1785686346
Terres du Midi        1785683822
Artisan               1785676720
Re snack              1785676645
BJORG                 1785675249
SEEBERGER             1785673739
Name: created_t, dtype: int64